# Part E — Determinants

**Final Project: Philippine Peso Depreciation Early-Warning Model: Predicting the Next-Month PHP/USD Exchange Rate Using Linear Regression and PCA**

Before you can solve for regression coefficients using matrix inversion, you need to know the inversion is actually possible.

---
**Student name:** Alfajaro, Ghan Vincent | Arile, Charles Jullianne | Asuar, Jhoe Maven | Mondragon, Leneth | Ulitin, Justin |   **Date:** November 16, 2026

---


**What to do:** Compute det(XᵀX) on your training data (after your Part D transformation) before attempting to invert it.

In [1]:
# Run this first: recall variables saved from Part D's notebook.
# (If this is your first notebook in the session, or the store is empty,
#  it fetches directly from the official BSP URL and applies Part D's exact transformation.)
%store -r X_train X_test y_train y_test mu sigma S

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [3]:
# Compute X_train^T X_train and its determinant on standardized Part D training data
XtX_train = X_train.T @ X_train
det_XtX_train = np.linalg.det(XtX_train)

print("X_train^T X_train (3x3):")
print(XtX_train)
print(f"\ndet(X_train^T X_train) = {det_XtX_train:.4f}")

X_train^T X_train (3x3):
[[342.     341.2789  35.8712]
 [341.2789 342.      26.2216]
 [ 35.8712  26.2216 342.    ]]

det(X_train^T X_train) = 135316.4004


In [4]:
# Decide, based on the determinant, whether XtX_train is invertible.
# Also compute its condition number and eigenvalues for a stability check.
cond_number = np.linalg.cond(XtX_train)
eigvals = np.linalg.eigvalsh(XtX_train)

if not np.isclose(det_XtX_train, 0.0):
    print("Conclusion: det(X_train^T X_train) != 0, so X_train^T X_train IS invertible.")
else:
    print("Conclusion: det(X_train^T X_train) is effectively 0, so X_train^T X_train is singular.")

print(f"Eigenvalues of X_train^T X_train: {eigvals}")
print(f"Condition number = {cond_number:.4f}")

Conclusion: det(X_train^T X_train) != 0, so X_train^T X_train IS invertible.
Eigenvalues of X_train^T X_train: [  0.5836 336.5783 688.838 ]
Condition number = 1180.2390


### Answer these in your report

**Q17.** What is det(XᵀX) for your training data?

>> *Your answer:* For our standardized chronological training set from Part D ($n_{\text{train}} = 342$ months from January 1991 to June 2019, $p = 3$ features), the determinant is $\det(X_{\text{train}}^T X_{\text{train}}) = 135,316.4004$. Because $X_{\text{train}}$ was standardized using `mu` and `sigma` from the training window, each diagonal entry of $X_{\text{train}}^T X_{\text{train}}$ equals $n_{\text{train}} = 342$, while the off-diagonal entries equal $342 \times r_{ij}$ (sample correlations scaled by $n_{\text{train}}$). Notice that while $\det(X_{\text{train}}^T X_{\text{train}})$ is clearly nonzero, it is much smaller than the uncorrelated benchmark of $342^3 = 40,001,688$ because `CurrentRate` ($X_1$) and `Trailing3MonthAverage` ($X_2$) are very strongly correlated ($r_{12} \approx 0.9979$).

**Q18.** Based on that value, is XᵀX invertible? If it were numerically singular, what would you do instead?

> *Your answer:* Yes, $X_{\text{train}}^T X_{\text{train}}$ is invertible because $\det(X_{\text{train}}^T X_{\text{train}}) \neq 0$, which confirms Part C's result that $X$ has full column rank ($3$). Therefore, $(X_{\text{train}}^T X_{\text{train}})^{-1}$ exists and the normal equations $(X_{\text{train}}^T X_{\text{train}})\beta = X_{\text{train}}^T y_{\text{train}}$ have a unique solution.
>
> If $X^T X$ were numerically singular ($\det(X^T X) \approx 0$), direct matrix inversion would fail or amplify floating-point rounding errors. Instead, we would:
> 1. **Drop or orthogonalize redundant features:** Remove one of the nearly collinear features (such as `Trailing3MonthAverage`) or apply **Principal Component Analysis (PCA)** so the transformed features are orthogonal and $X^T X$ becomes diagonal.
> 2. **Apply Ridge Regression ($L_2$ regularization):** Solve $(X^T X + \lambda I)\beta = X^T y$ with a small positive penalty $\lambda > 0$, which shifts all eigenvalues away from zero and guarantees invertibility.
> 3. **Use SVD / Moore-Penrose pseudoinverse (`np.linalg.pinv`):** Compute the minimum-norm least-squares solution directly via Singular Value Decomposition without forming $(X^T X)^{-1}$.

**Q19.** Describe a realistic situation in your own dataset (e.g., two nearly duplicate features) that could push this determinant close to zero.

> *Your answer:* In our BSP PHP/USD dataset, adding a **Trailing 2-Month Average** ($\frac{R_t + R_{t-1}}{2}$) alongside `CurrentRate` ($R_t$) and `Trailing3MonthAverage` ($\frac{R_t + R_{t-1} + R_{t-2}}{3}$) would push $\det(X^T X)$ very close to zero. Because monthly exchange rates move smoothly from one month to the next, $R_t$, the 2-month average, and the 3-month average track each other almost identically, making the new column nearly a linear combination of the existing two. Similarly, including both the **Year-over-Year peso change** ($R_t - R_{t-12}$) and the **Year-over-Year percentage change** ($\frac{R_t - R_{t-12}}{R_{t-12}}$) would create near-duplicate columns and collapse the determinant toward zero.

**Q20.** Look up or compute the condition number of XᵀX. How does a large condition number relate to the determinant, and what does it suggest about the stability of your model?

> > *Your answer:* The condition number of $X_{\text{train}}^T X_{\text{train}}$ is $\kappa(X_{\text{train}}^T X_{\text{train}}) = 1,180.2390$, which equals the ratio of its largest eigenvalue to its smallest eigenvalue ($\lambda_{\max} / \lambda_{\min} \approx 688.8380 / 0.5836$). While the determinant is the **product** of all eigenvalues ($\det(X^T X) = \lambda_1 \lambda_2 \lambda_3 \approx 0.5836 \times 336.5783 \times 688.8380 = 135,316.4004$), a large condition number reveals that the smallest eigenvalue $\lambda_{\min} \approx 0.5836$ is tiny compared to $\lambda_{\max} \approx 688.8380$—meaning the matrix is close to singular along one direction even when its determinant is still well above zero.
>
> For our regression model, this large condition number reflects the high collinearity between `CurrentRate` and `Trailing3MonthAverage` that we saw in Parts B and C. It tells us that although $(X^T X)^{-1}$ exists, the individual coefficient estimates ($\hat{\beta}_1$ and $\hat{\beta}_2$) are moderately sensitive to small changes or noise in the training data, even though overall predictions remain accurate.

### Save your work for the next notebook

In [5]:
%store XtX_train
%store X_train
%store X_test
%store y_train
%store y_test
%store mu
%store sigma
%store S
print("Saved XtX_train (and Part D train/test splits & scaling parameters) for Part F.")

Stored 'XtX_train' (ndarray)
Stored 'X_train' (ndarray)
Stored 'X_test' (ndarray)
Stored 'y_train' (ndarray)
Stored 'y_test' (ndarray)
Stored 'mu' (ndarray)
Stored 'sigma' (ndarray)
Stored 'S' (ndarray)
Saved XtX_train (and Part D train/test splits & scaling parameters) for Part F.
